# Distillation từ model thầy so với GRPO và tự học (rejection sampling)

Ba cách làm model nhỏ giỏi toán hơn mà module đã dựng: tự học từ lời giải đúng của chính nó (Intermediate 03), RL với
reward kiểm được (Advanced 01), và **distillation**: model thầy lớn hơn sinh lời giải, lọc đúng, SFT model trò. Bài này
dựng nhánh thứ ba và so cả ba trên cùng 500 câu test ghim, cùng bộ đo, có khoảng tin cậy. Viết kết luận vào `REPORT.md`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng, hàm. Điền vào chỗ `# TODO`.
- Cell `explore` là khung gợi ý: sửa thoải mái.
- Chạy bằng kernel `ai-gpu`. Model thầy: `Qwen2.5-3B-Instruct` qua Transformers (cần khoảng 8 GB VRAM, PC) hoặc
  `qwen2.5:3b` / `qwen2.5:7b` qua Ollama (laptop dùng `qwen2.5:3b`).
- Chấm: `uv run pytest tracks/ai-engineer/modules/15-reasoning-models/exercises/advanced/02-distill-vs-grpo -q`

In [ ]:
# explore: tìm gốc repo, tải GSM8K nếu chưa có
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

GSM = ensure("gsm8k")
CACHE = GSM / "cache"
CACHE.mkdir(exist_ok=True)

In [ ]:
import re
from fractions import Fraction

import numpy as np
import pandas as pd

## 1. Dán và viết bảng so sánh

Dán từ các bài trước: `normalize_number`, `extract_answer`, `format_reward`, `correctness_reward` (Basic 01),
`paired_bootstrap` (Intermediate 01), `select_traces`, `to_sft_record` (Intermediate 03). Rồi viết:

`compare_table(results, base="base") -> DataFrame`: `results` là dict `tên phương pháp -> kết quả đo`, mỗi kết quả có
`accuracy`, `boxed_correct`, `format`, `mean_tokens` và `correct` (list `bool` từng câu của tập đo). Trả một dòng mỗi
phương pháp theo thứ tự của dict, cột đúng thứ tự `method, accuracy, boxed_correct, format, mean_tokens, delta, lo, hi`:
`delta, lo, hi` là `paired_bootstrap(results[base]["correct"], results[m]["correct"], seed=0)` (dòng của `base` là
`0, 0, 0`). Danh sách `correct` khác độ dài với của `base` (đo trên tập khác) thì `ValueError`; thiếu `base` thì `KeyError`.

In [ ]:
SYSTEM_COT = "Please reason step by step, and put your final answer within \\boxed{}."


def normalize_number(text: str) -> str | None:
    """Dán từ bài Basic 01 (cùng mọi regex và hàm phụ nó cần)."""
    # TODO
    raise NotImplementedError


def extract_answer(completion: str) -> str | None:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def format_reward(completion: str) -> float:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def correctness_reward(completion: str, gold: str) -> float:
    """Dán từ bài Basic 01."""
    # TODO
    raise NotImplementedError


def paired_bootstrap(correct_a, correct_b, n_boot: int = 2000, seed: int = 0) -> tuple[float, float, float]:
    """Dán từ bài Intermediate 01."""
    # TODO
    raise NotImplementedError


def select_traces(samples: list[str], gold: str, max_keep: int = 2) -> list[str]:
    """Dán từ bài Intermediate 03."""
    # TODO
    raise NotImplementedError


def to_sft_record(question: str, trace: str, system: str = SYSTEM_COT) -> dict:
    """Dán từ bài Intermediate 03."""
    # TODO
    raise NotImplementedError


def compare_table(results: dict[str, dict], base: str = "base") -> pd.DataFrame:
    """Một dòng mỗi phương pháp, hiệu accuracy so với base kèm khoảng tin cậy theo cặp."""
    # TODO
    raise NotImplementedError

## 2. Sinh lời giải bằng model thầy

Cùng 800 câu train (trộn theo seed 0) như Intermediate 03, 2 lời giải mỗi câu ở `temperature` 0.7, cùng prompt CoT.
Đo luôn model thầy greedy trên 500 câu test để biết trần của nó. Laptop: Ollama `qwen2.5:3b` (khoảng 1 600 lượt gọi,
có thể giảm số câu và ghi rõ); PC: Transformers bf16 (nhanh hơn nhiều nhờ batch).

In [ ]:
# explore: TODO: sinh, cache ra CACHE / "m15-distill-teacher.json"

## 3. Lọc, SFT model trò, đo

Cùng công thức SFT với Intermediate 03 (LoRA r = 16, 2 epoch, lr 1e-4) để chỉ khác **nguồn dữ liệu**. Đo greedy trên 500
câu test ghim với cùng hàm đo.

In [ ]:
# explore: TODO: select_traces, to_sft_record, SFTTrainer, evaluate

## 4. So sánh

Đọc lại kết quả đã cache của các bài trước (model gốc, Intermediate 03, Advanced 01 theo seed) và của bài này, đưa vào
`compare_table`. Thêm cột chi phí bạn tự đo: thời gian sinh dữ liệu, thời gian train, VRAM đỉnh.

In [ ]:
# explore: TODO: results = {"base": ..., "rft (tự học)": ..., "distill 3B": ..., "grpo seed 0": ..., ...}
# display(compare_table(results))